# Kidney dataset selection — audit (`_repaired`)

**v2 — przepisane na właściwe funkcje biblioteczne.** Poprzednia wersja tego notebooka implementowała detekcję duplikatów/QC/morfologii jako kod inline w komórkach (pandas/regex). Od `packages/msi_dataset_manager/src/msi_dataset_manager/exploration/dataset_review.py` ta logika istnieje jako właściwy moduł biblioteczny: `DatasetReview`, `DatasetReviewProfile`, `DatasetExplorer.review_current()`, `DatasetExplorer.apply_review()`. Ten notebook używa teraz wyłącznie tych funkcji — zero zduplikowanej logiki.

**Ograniczenie, które zostaje:** wbudowane profile w bibliotece to na dziś tylko `"brain"` i `"liver"` (`_PROFILES` w `dataset_review.py`) — **nie ma profilu `"kidney"`**. Zamiast go dopisywać do biblioteki (nie modyfikuję jej), przekazuję niżej równoważny `DatasetReviewProfile` bezpośrednio z poziomu notebooka — dokładnie te same reguły, których używałem poprzednio ręcznie dla kidney (`cortex`/`medulla`/`papilla`/`pelvis`/`calyx`/`glomerul`, bez progu niskiej liczby pikseli). Jeśli chcesz, żeby `profile="kidney"` działało tak samo jak `"brain"`/`"liver"`, dopisz `_KIDNEY_PROFILE` do `_PROFILES` w `dataset_review.py` z tymi samymi wartościami.

Świadomie bez zmian względem poprzedniej wersji: brak analizy wspólnego zakresu m/z między narządami (do rozwiązania osobno), brak zmian w bibliotece, brak nowych pobrań surowych danych.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import json

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer, DatasetReviewProfile

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as brain_dataset.ipynb.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów (ten sam filtr biologiczny co dotychczasowy `kidney_dataset.ipynb`)

Bez zmian względem poprzedniej wersji: `condition="Wildtype"` (dokładnie filtr, który wygenerował obecny korpus kidney), bez `mz_min`/`mz_max`, żeby audyt objął też rekordy odrzucane dziś przez filtr `200–900`.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Kidney",
    "condition": "Wildtype",
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "analyzer_type", "ionisation_source", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 60 datasets


,dataset_id,name,analyzer_type,ionisation_source,mz_min,mz_max,pixel_count
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,Orbitrap,MALDI,90.002395,999.994556,8575
1,2025-04-14_15h53m53s,00070_lgruber_qcl-msi_data11_glomeruli_wt_rms,timsTOF fleX,MALDI,300.015000,2500.000000,11057
2,2025-04-14_09h06m15s,00070_lgruber_qcl-msi_het-0717_data3_slide1-rms,timsTOF fleX,MALDI,300.006000,1999.970000,39149
3,2025-04-14_09h00m43s,00070_lgruber_qcl-msi_het-0707_data5_slide1-rms,timsTOF fleX,MALDI,300.006000,1999.980000,39021
4,2025-04-14_15h55m50s,00070_lgruber_qcl-msi_timson_glomeruli_data2_w...,timsTOF fleX,MALDI,300.010500,2500.000000,10163
5,2025-04-14_09h56m19s,00070_lgruber_qcl-msi_wt-4625_data6_slide1-rms,timsTOF fleX,MALDI,300.007500,1999.980000,47682
6,2025-04-14_10h18m43s,00070_lgruber_qcl-msi_wt-4628_data1_slide1-rms,timsTOF fleX,MALDI,300.007500,1999.990000,44832
7,2025-07-07_13h56m06s,mo6_metabolites,timsTOF fleX,MALDI,50.001500,649.990250,184011
8,2025-07-07_13h03m27s,msi2024013_20240909_multiomicsiii_nor_300-1350...,timsTOF fleX,MALDI,300.004500,1349.979750,83047
9,2025-07-07_10h55m54s,msi2024013_20240909_multiomicsii_nedc 50-650_f...,timsTOF fleX,MALDI,50.001500,649.990250,87617


## 2. Ładowanie dotychczasowej ręcznej selekcji i przeniesienie jej wykluczeń do sesji

`data/kidney_workspace/configs/datasets/kidney/filter.json` ma już 30 ręcznie wykluczonych ID z poprzedniej rundy przeglądu (przy tej samej puli 60 kandydatów: 30 zaakceptowanych + 30 wykluczonych = 60). Przenoszę je do sesji `explorer.exclude(...)` **przed** przeglądem obiektywnym, żeby finalne wykluczenia w kroku 4 były unią obu (dotychczasowe ręczne ∪ nowo znalezione obiektywne), a nie tylko nowych.

In [4]:
existing_filter = json.load(open("data/kidney_workspace/configs/datasets/kidney/filter.json"))
existing_selection = json.load(open("data/kidney_workspace/configs/datasets/kidney/selection.json"))
existing_excluded_ids = existing_filter.get("exclude_dataset_ids", [])
existing_selected_ids = set(existing_selection["dataset_ids"])
print(f"existing selection: {len(existing_selected_ids)} selected, {len(existing_excluded_ids)} manually excluded")

explorer.exclude(existing_excluded_ids)

existing selection: 30 selected, 30 manually excluded


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Kidney,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False
1,2025-07-06_22h07m04s,msi2024013_20240909_multiomicsii_nor 50-1350,metaspace,None,https://metaspace2020.eu/dataset/2025-07-06_22...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2025-07-06_16h31m32s,msi2024013_20240909_multiomicsii_nedc 50-1350,metaspace,None,https://metaspace2020.eu/dataset/2025-07-06_16...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
3,2024-05-23_14h25m01s,K3 vs K6 neg -Jano,metaspace,None,https://metaspace2020.eu/dataset/2024-05-23_14...,Mouse,Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
4,2024-04-10_17h06m19s,8223301,metaspace,None,https://metaspace2020.eu/dataset/2024-04-10_17...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
5,2024-02-20_01h55m56s,d28-2014-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
6,2024-02-20_01h57m32s,d28-2017-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
7,2024-02-20_01h54m01s,d14-2295,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
8,2024-02-20_01h54m41s,d28-2006-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
9,2024-02-20_01h53m31s,d14-2015,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False


## 3. Przegląd biblioteczny (`DatasetExplorer.review_current`)

Jedno wywołanie zastępuje całą poprzednią, ręcznie pisaną sekcję 2/2b/3: obiektywne klastry duplikatów technicznych (`pixel_count`+`mz_min`+`mz_max`, z rozróżnieniem `high_confidence_duplicate` vs `ambiguous_shared_template`), wariant kalibracyjny `null_mz_shift`, heurystykę morfologii, oraz — nowość, której nie miałem ręcznie — `biological_series_id` liczone tą samą metodą co w `brain_dataset.ipynb`.

In [5]:
kidney_profile = DatasetReviewProfile(
    low_pixel_threshold=None,  # kidney's pool minimum is 5025 px -- no outliers to flag, see prior audit
    morphology_pattern=r"(?:cortex|medulla|papilla|pelvis|calyx|glomerul)",
    explicit_regional_names=frozenset(),  # no user-confirmed explicit fragments for kidney (unlike brain)
)
review = explorer.review_current(profile=kidney_profile)

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ]
)
display(
    review.table.loc[
        review.table["mz_shift_qc_variant"] | review.table["morphology_hint"].eq("regional_or_microregion"),
        ["dataset_id", "name", "pixel_count", "mz_shift_qc_variant", "morphology_hint"],
    ]
)

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,0
1,mz_shift_qc_variants,1
2,explicit_regional_fragments,0


,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id


,dataset_id,name,pixel_count,mz_shift_qc_variant,morphology_hint
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,8575,True,whole_section_likely


## 4. Zastosowanie reguł i finalna, poprawiona lista

Stosuję `high_confidence_duplicates` i `mz_shift_qc_variants` — te same reguły, co poprzednio uznałem za wystarczająco pewne, żeby wykluczać automatycznie. `explicit_regional_fragments` też wywołuję (zwraca 0 dla kidney, bo `explicit_regional_names` jest puste — patrz sekcja 3) — dla spójności z tym, jak robi to `brain_dataset.ipynb`. Heurystyczne `morphology_hint` (kolumna doradcza, `glomeruli`) **nie** jest regułą do zastosowania — zostaje tylko w tabeli do Twojego przeglądu.

In [6]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Kidney",
    "polarity": "Negative",
    "condition": "Wildtype",
    "mz_min": 200,
    "mz_max": 900,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# NOTE: exclude_dataset_ids is intentionally omitted here -- explorer.exclude()/apply_review()
# already marked the union of prior-manual and newly-reviewed IDs as excluded on this session,
# and that state persists across this re-query (verified: results() always re-applies it).
results_kidney_repaired = explorer.filter(final_filters)
print(f"repaired kidney shortlist: {len(results_kidney_repaired)} datasets (previously {len(existing_selected_ids)})")
display(results_kidney_repaired[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

repaired kidney shortlist: 29 datasets (previously 30)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2025-07-06_22h07m04s,msi2024013_20240909_multiomicsii_nor 50-1350,timsTOF fleX,83704,103,44
1,2025-07-06_16h31m32s,msi2024013_20240909_multiomicsii_nedc 50-1350,timsTOF fleX,70784,23,13
2,2024-05-23_14h25m01s,K3 vs K6 neg -Jano,Orbitrap,20096,311,221
3,2024-04-10_17h06m19s,8223301,Orbitrap,20992,124,69
4,2024-02-20_01h55m56s,d28-2014-2,FTICR,7198,7,0
5,2024-02-20_01h57m32s,d28-2017-2,FTICR,7664,14,6
6,2024-02-20_01h54m01s,d14-2295,FTICR,5025,10,7
7,2024-02-20_01h54m41s,d28-2006-2,FTICR,8566,14,1
8,2024-02-20_01h53m31s,d14-2015,FTICR,6697,14,6
9,2024-02-20_01h49m35s,d14-2001-2,FTICR,8278,11,1


## 5. Które datasety zostały wycięte względem dotychczasowego `kidney/` — pełne porównanie

Jedna tabela: dla każdego z 60 kandydatów pokazuje, czy był w dotychczasowej selekcji (30), czy jest w poprawionej (`repaired`), i **dlaczego** coś się zmieniło — konkretna reguła, nie tylko "wykluczony/nie".

In [7]:
repaired_selected_ids = set(results_kidney_repaired["dataset_id"].astype(str))
review_reasons = {
    dataset_id: "high_confidence_duplicate"
    for dataset_id in review.exclusion_ids(["high_confidence_duplicates"])
}
review_reasons.update({
    dataset_id: "mz_shift_qc_variant"
    for dataset_id in review.exclusion_ids(["mz_shift_qc_variants"])
})

comparison = results[["dataset_id", "name"]].copy()
comparison["in_original_selection"] = comparison["dataset_id"].isin(existing_selected_ids)
comparison["in_repaired_selection"] = comparison["dataset_id"].isin(repaired_selected_ids)
comparison["was_manually_excluded_before"] = comparison["dataset_id"].isin(existing_excluded_ids)
comparison["newly_excluded_reason"] = comparison["dataset_id"].map(review_reasons).fillna("")

changed = comparison.loc[comparison["in_original_selection"] != comparison["in_repaired_selection"]]
print(f"datasets whose accept/exclude status changed: {len(changed)}")
display(changed)

print(f"\ntotal: {comparison['in_original_selection'].sum()} (original) -> {comparison['in_repaired_selection'].sum()} (repaired)")

datasets whose accept/exclude status changed: 1


,dataset_id,name,in_original_selection,in_repaired_selection,was_manually_excluded_before,newly_excluded_reason
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,True,False,False,mz_shift_qc_variant



total: 30 (original) -> 29 (repaired)


In [8]:
output_path = Path("data/kidney_workspace/configs/datasets/kidney_repaired")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/kidney_workspace/configs/datasets/kidney_repaired/filter.json'),
 'selection': PosixPath('data/kidney_workspace/configs/datasets/kidney_repaired/selection.json')}

## Podsumowanie

- Przepisane na `DatasetExplorer.review_current()`/`.apply_review()` — bez duplikowania logiki w komórkach.
- Kidney nie ma jeszcze wbudowanego profilu w bibliotece (`_PROFILES` ma tylko `brain`/`liver`) — używam równoważnego `DatasetReviewProfile` z poziomu notebooka; jeśli dopiszesz `_KIDNEY_PROFILE` do `dataset_review.py`, ten notebook można uprościć do `profile="kidney"`.
- Zmiana względem dotychczasowej selekcji: **1 dataset wycięty** (`kidney_test_metabolites_null_mz_shift_10_til_550`, wariant kalibracyjny obecny w pobranym korpusie, umknął wcześniejszej ręcznej rundzie) → 30 → 29.
- Pełna tabela zmian w sekcji 5.
- Eksport do `data/kidney_workspace/configs/datasets/kidney_repaired/` — istniejący `kidney/` nie został nadpisany.